# 資料前處理(Label encoding、 One hot encoding)
這兩個編碼方式的目的是為了將類別 (categorical)或是文字(text)的資料轉換成數字，而讓程式能夠更好的去理解及運算。
> Label encoding : 把每個類別 mapping 到某個整數，不會增加新欄位

> One hot encoding : 為每個類別新增一個欄位，用 0/1 表示是否

![](images/Encoder.PNG)


## Encoding Categorical features (or label)
![](images/Encoding.PNG)


In [6]:
import pandas as pd
import numpy as np


In [7]:
df = pd.DataFrame({'blood':['A','B','AB','O','B'],

                   'Y':['high','low','high','mid','mid'],

                   'Z':[np.nan, np.nan, -1196.7, 2.83, np.nan]});

df

,blood,Y,Z
0,A,high,NaN
1,B,low,NaN
2,AB,high,-1196.70
3,O,mid,2.83
4,B,mid,NaN


# 方法一：sklearn - label encoder + onehot encoder
>onehot encoder要用2D array，若維度所以要用reshape(-1,1)<br>
>onehot encoder要數字，若資料文文字要先用label encoder轉數字

In [8]:
from sklearn.preprocessing import LabelEncoder

 

encoder = LabelEncoder()

encoded_Y = encoder.fit_transform(df['blood'])

print(encoded_Y)

df['blood'] = encoded_Y

df

[0 2 1 3 2]


,blood,Y,Z
0,0,high,NaN
1,2,low,NaN
2,1,high,-1196.70
3,3,mid,2.83
4,2,mid,NaN


In [9]:
from sklearn.preprocessing import OneHotEncoder

 

onehot = OneHotEncoder()

d = np.array(df['blood'])

d.shape

onehot_df = onehot.fit_transform(d.reshape(-1, 1)).toarray()

onehot_df

 

array([[1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.]])

## One hot encoding
One Hot encoding的編碼邏輯為將類別拆成多個行(column)，每個列中的數值由1、0替代，當某一列的資料存在的該行的類別則顯示1，反則顯示0。

然在指定column進行編碼的情形下，One hot encoding<b>無法直接對字串進行編碼，必須先透過Label encoding將字串以數字取代後再進行One hot encoding處理。</b>

> categorical_features = [0]: 表示欲在data上執行One hot encoding的index為0

> data_le: 為經過Label encoding編碼的資料(註:OneHotEncoder的輸入要為2-D array，而Label encoding為1-D array)


OneHotEncoder會轉出scipy.csr_matrix資料結構用.toarray()轉array
從結果可以知道，數字0的column 代表的是A、數字1的column 代表的是B，而數字2的column 代表的是AB。
除了轉換字串外，One hot encoding也可以轉換數字。在此處的data就不需要先經過Label encoding編碼

```python
# importing one hot encoder from sklearn 
# There are changes in OneHotEncoder class 
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer 
   
# creating one hot encoder object with categorical feature 0 
# indicating the first column 
columnTransformer = ColumnTransformer([('encoder', 
                                        OneHotEncoder(), 
                                        [0])], 
                                      remainder='passthrough') 
  
data = np.array(columnTransformer.fit_transform(data), dtype = str) 
```

In [10]:
# importing one hot encoder from sklearn

# There are changes in OneHotEncoder class

from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer

 

# creating one hot encoder object with categorical feature 0

# indicating the first column

columnTransformer = ColumnTransformer([('encoder',

                                        OneHotEncoder(),

                                        [0])],

                                      remainder='passthrough')

 

data = np.array(columnTransformer.fit_transform(df), dtype = str)

data

 

data_le = pd.DataFrame(data)

data_le

 


,0,1,2,3,4,5
0,1.0,0.0,0.0,0.0,high,nan
1,0.0,0.0,1.0,0.0,low,nan
2,0.0,1.0,0.0,0.0,high,-1196.7
3,0.0,0.0,0.0,1.0,mid,2.83
4,0.0,0.0,1.0,0.0,mid,nan


# 方法二：Keras - label encoder + to_categorical
>to_categorical要數字，若資料文文字要先用label encoder轉數字

In [11]:
!pip install tensorflow


In [12]:

from sklearn.preprocessing import LabelEncoder

from tensorflow.keras import utils as np_utils

 

df = pd.DataFrame({'blood':['A','B','AB','O','B'],

                   'Y':['high','low','high','mid','mid'],

                   'Z':[np.nan, np.nan, -1196.72, 83, np.nan]});

 

# label encoder

encoder = LabelEncoder()

encoded_Y = encoder.fit_transform(df['blood'])

print(encoded_Y)

df['blood'] = encoded_Y

df

 

# convert integers to one hot encoding

keras_onehot = np_utils.to_categorical(encoded_Y)

keras_onehot 



[0 2 1 3 2]


array([[1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.]])

## 方法三：pd.get_dummies方法
![](images/Encoding_pd.PNG)
pd.get_dummies(df)
>get_dummies可以直接轉字串，反而無法轉換數字<br>
>get_dummies沒指定columns，會全部轉換

In [13]:
df = pd.DataFrame({'blood':['A','B','AB','O','B'],

                   'Y':['high','low','high','mid','mid'],

                   'Z':[np.nan, np.nan, -1196.7, 2.83, np.nan]})

 

# 2. 對整個 DataFrame 進行 One-Hot 編碼

# 它會自動找文字型欄位（blood, Y）來轉換，數值型（Z）會保留

df1 = pd.get_dummies(df)

print(df1)

 

# 3. 只針對單一欄位（blood）進行 One-Hot 編碼

df2 = pd.get_dummies(df.blood)

print(df2)

         Z  blood_A  blood_AB  blood_B  blood_O  Y_high  Y_low  Y_mid
0      NaN     True     False    False    False    True  False  False
1      NaN    False     False     True    False   False   True  False
2 -1196.70    False      True    False    False    True  False  False
3     2.83    False     False    False     True   False  False   True
4      NaN    False     False     True    False   False  False   True
       A     AB      B      O
0   True  False  False  False
1  False  False   True  False
2  False   True  False  False
3  False  False  False   True
4  False  False   True  False


## 練習一：sklearn - label encoder + onehot encoder
下面的資料可以看到country那欄皆為字串， 大部分的模型都是基於數學運算，字串無法套入數學模型進行運算，<br>
在此先對其進行Label encoding編碼，我們從 sklearn library中導入 LabelEncoder class，對第一行資料進行fit及transform並取代之。

In [14]:
import numpy as np

import pandas as pd

from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from sklearn.compose import ColumnTransformer

 

# 1. 建立原始資料 (對應圖片中的練習一內容)

country = ['Taiwan', 'Australia', 'Ireland', 'Australia', 'Ireland', 'Taiwan']

age = [25, 30, 45, 35, 22, 36]

salary = [20000, 32000, 59000, 60000, 43000, 52000]

 

dic = {'Country': country, 'Age': age, 'Salary': salary}

data = pd.DataFrame(dic)

print("--- 原始資料 ---")

print(data)

print("\n")

 

# 2. 步驟一：使用 Label Encoding 將國家文字轉為數字

labelencoder = LabelEncoder()

data['Country'] = labelencoder.fit_transform(data['Country'])

print("--- 經過 Label Encoding (文字轉數字) ---")

print(data)

print("\n")

 

# 3. 步驟二：使用 One-Hot Encoding 展開欄位 (避免數字大小影響模型)

# 我們指定對索引為 0 的欄位 (Country) 進行 One-Hot 編碼

ct = ColumnTransformer([('encoder', OneHotEncoder(), [0])], remainder='passthrough')

 

# 執行轉換並轉成 NumPy 陣列，再轉回 DataFrame 方便閱讀

data_final = pd.DataFrame(ct.fit_transform(data))

 

# 重新命名欄位，讓表格更清楚 (前三欄是國家，後兩欄是原有的 Age 和 Salary)

data_final.columns = ['Australia', 'Ireland', 'Taiwan', 'Age', 'Salary']

 

print("--- 最終結果 (One-Hot Encoding) ---")

print(data_final)

--- 原始資料 ---
     Country  Age  Salary
0     Taiwan   25   20000
1  Australia   30   32000
2    Ireland   45   59000
3  Australia   35   60000
4    Ireland   22   43000
5     Taiwan   36   52000


--- 經過 Label Encoding (文字轉數字) ---
   Country  Age  Salary
0        2   25   20000
1        0   30   32000
2        1   45   59000
3        0   35   60000
4        1   22   43000
5        2   36   52000


--- 最終結果 (One-Hot Encoding) ---
   Australia  Ireland  Taiwan   Age   Salary
0        0.0      0.0     1.0  25.0  20000.0
1        1.0      0.0     0.0  30.0  32000.0
2        0.0      1.0     0.0  45.0  59000.0
3        1.0      0.0     0.0  35.0  60000.0
4        0.0      1.0     0.0  22.0  43000.0
5        0.0      0.0     1.0  36.0  52000.0


## 練習二：Keras - label encoder + to_categorical

In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# 1. 準備練習一的原始資料
country = ['Taiwan', 'Australia', 'Ireland', 'Australia', 'Ireland', 'Taiwan']
age = [25, 30, 45, 35, 22, 36]
salary = [20000, 32000, 59000, 60000, 43000, 52000]

dic = {'Country': country, 'Age': age, 'Salary': salary}
data = pd.DataFrame(dic)

# 2. 步驟一：使用 Label Encoding (文字轉整數)
# 這是 Keras 處理前必須做的準備動作
label_encoder = LabelEncoder()
integer_encoded = label_encoder.fit_transform(data['Country'])

print("整數編碼結果:", integer_encoded)
# 結果會是類似 [2 0 1 0 1 2] (依字母排序)

# 3. 步驟二：使用 Keras 的 to_categorical (整數轉 One-hot)
one_hot_encoded = to_categorical(integer_encoded)

# 4. 合併結果
# 將 One-hot 矩陣與原本的 Age, Salary 合併
final_result = np.column_stack((one_hot_encoded, data['Age'], data['Salary']))

print("\n--- 練習二結果 (Keras 方式) ---")
print("前三欄為國家 One-hot，後兩欄為年齡與薪水：")
print(final_result)

整數編碼結果: [2 0 1 0 1 2]

--- 練習二結果 (Keras 方式) ---
前三欄為國家 One-hot，後兩欄為年齡與薪水：
[[0.0e+00 0.0e+00 1.0e+00 2.5e+01 2.0e+04]
 [1.0e+00 0.0e+00 0.0e+00 3.0e+01 3.2e+04]
 [0.0e+00 1.0e+00 0.0e+00 4.5e+01 5.9e+04]
 [1.0e+00 0.0e+00 0.0e+00 3.5e+01 6.0e+04]
 [0.0e+00 1.0e+00 0.0e+00 2.2e+01 4.3e+04]
 [0.0e+00 0.0e+00 1.0e+00 3.6e+01 5.2e+04]]


## 練習三：Pandas.get_dummies
>　get_dummies : 僅能將字串轉換為One hot encoding表示形式， 沒指定columns會全部轉換。

In [16]:
import pandas as pd

 

# 1. 建立資料 (跟前面練習一樣)

country = ['Taiwan', 'Australia', 'Ireland', 'Australia', 'Ireland', 'Taiwan']

age = [25, 30, 45, 35, 22, 36]

salary = [20000, 32000, 59000, 60000, 43000, 52000]

dic = {'Country': country, 'Age': age, 'Salary': salary}

data = pd.DataFrame(dic)

 

# 2. 使用 Pandas get_dummies (練習三的重點)

# 只要這一行，它會自動把資料表中的「字串」欄位轉成 One-Hot 形式

data_dummies = pd.get_dummies(data)

 

# 顯示結果

print(data_dummies)

 

   Age  Salary  Country_Australia  Country_Ireland  Country_Taiwan
0   25   20000              False            False            True
1   30   32000               True            False           False
2   45   59000              False             True           False
3   35   60000               True            False           False
4   22   43000              False             True           False
5   36   52000              False            False            True
